# Mixed-Type IL OpTC -> ATLASv2 (từ điển mở rộng + mở dần + replay OpTC thật)

Incremental learning **OpTC -> ATLASv2** cho mô hình B (4 category + `delta_t`; đã bỏ field Bro và `principal_class`), có **từ điển mở rộng**, **replay OpTC thật**, **mở dần đóng băng** (freezing/gradual unfreezing) và **VAL riêng** để chọn checkpoint/ngưỡng. Thiết kế ở `02_artifacts/44_freezing_design_v1/STEP_44_FREEZING_DESIGN.md` và `02_artifacts/48_il_atlasv2_dataset_v1/STEP_48_IL_ATLASV2_DATASET.md`.

**Gắn đúng 2 dataset:** (1) `v2_network_dl_kaggle_upload` (OpTC v2: replay và đánh giá OpTC TEST); (2) `il_atlasv2_kaggle_upload` (gói ATLASv2, ~7 MB). Bật GPU.

Mỗi fold (3 fold, chia trộn theo cửa sổ, seed 42): kiểm tra khởi tạo tương đương mô hình gốc; giai đoạn **A** (hàng embedding mới + đầu phân loại) -> **B1** (+ encoder sự kiện) -> **B2** (+ tầng chuỗi) -> **C** (+ embedding cũ, lr nhỏ); mỗi giai đoạn dừng sớm theo VAL; checkpoint và ngưỡng chỉ chọn trên VAL; TEST và OpTC TEST tính **một lần**. Mixed-Type: lr giai đoạn A 1e-3; B1/B2/C lr cơ sở 1e-4 (hệ số theo nhóm), batch 16; các giá trị này chưa được tinh chỉnh.

**Cảnh báo khi đọc kết quả:** nhãn ATLASv2 là nhãn REAPr mức **tiến trình** (firefox có cả 0 và 1); chỉ ~59 cửa sổ dương, 10 cuộc tấn công gần giống nhau; chia trộn theo cửa sổ nên cửa sổ kề cùng cuộc tấn công nằm ở các phần khác nhau (rò rỉ có chủ ý) => điểm cao đo khả năng nhận ra mẫu đã thấy, không đo phát hiện tấn công mới. Không có so sánh trước/sau fine-tune; chỉ có ma trận nhầm lẫn TRAIN, VAL, TEST (ATLASv2) và OpTC TEST.

In [ ]:
import os, sys, json, shutil
from pathlib import Path
import torch
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))


## Cấu hình
Một seed (42), 3 fold. Đổi `FOLDS` để chạy ít hơn (ví dụ `'0'`). `OPTC_EVAL='none'` bỏ đánh giá OpTC TEST (nhanh hơn).

In [ ]:
MODEL = 'mixed'
FOLDS = '0,1,2'
SEED = 42
OPTC_EVAL = 'test'
RUN_NAME = f'{MODEL}_il_atlasv2_seed{SEED}'
print(MODEL, FOLDS, SEED, RUN_NAME)


## Tìm hai dataset

In [ ]:
from __future__ import annotations

import shutil
from pathlib import Path


def unwrap_same_name_folder(
    root: Path,
    name: str,
) -> Path:
    path = root / name

    if not path.exists():
        raise FileNotFoundError(
            f"Missing folder: {path}"
        )

    while (path / name).is_dir():
        path = path / name

    return path


def resolve_kaggle_dataset_root(
    *,
    input_root: str | Path = "/kaggle/input",
    working_root: str | Path = "/kaggle/working/optc_network_dl",
) -> Path:
    input_root = Path(input_root)

    manifests = list(
        input_root.rglob(
            "dataset_manifest.json"
        )
    )

    if not manifests:
        raise FileNotFoundError(
            "Cannot find dataset_manifest.json under /kaggle/input"
        )

    if len(manifests) > 1:
        raise RuntimeError(
            "Nhieu dataset_manifest.json duoc tim thay (vd. dataset LOSO co 3 holdout). "
            "Chi gan MOT dataset vao notebook:\n"
            + "\n".join(str(m) for m in manifests)
        )

    source_root = None

    for manifest in manifests:
        candidate = manifest.parent

        if (
            candidate
            / "build_config.json"
        ).exists():
            source_root = candidate
            break

    if source_root is None:
        raise RuntimeError(
            "Found dataset_manifest.json but not build_config.json"
        )

    working_root = Path(working_root)

    if working_root.exists():
        shutil.rmtree(working_root)

    working_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    for filename in (
        "dataset_manifest.json",
        "build_config.json",
        "README_DATASET.md",
    ):
        src = source_root / filename

        if src.exists():
            (
                working_root / filename
            ).symlink_to(src)

    for folder in (
        "shards",
        "split",
        "preprocessing",
        "integrity",
    ):
        real_folder = unwrap_same_name_folder(
            source_root,
            folder,
        )

        (
            working_root / folder
        ).symlink_to(
            real_folder,
            target_is_directory=True,
        )

    required = [
        "dataset_manifest.json",
        "build_config.json",
        "split/network_sample_manifest.csv",
        "preprocessing/network_category_maps.json",
        "preprocessing/network_numeric_stats.json",
        "preprocessing/network_preprocess_spec.json",
        "preprocessing/network_preprocess_fingerprint.json",
        "shards/train",
        "shards/val",
        "shards/test",
    ]

    for relative in required:
        path = working_root / relative

        print(
            f"{'PASS' if path.exists() else 'FAIL':4s}",
            relative,
        )

        if not path.exists():
            raise FileNotFoundError(path)

    return working_root


DATASET_ROOT = resolve_kaggle_dataset_root()
print('\nDATASET_ROOT =', DATASET_ROOT)
manifest = json.loads((DATASET_ROOT/'dataset_manifest.json').read_text())
print(json.dumps(manifest['split_stats'], indent=2))


In [ ]:
ft_manifests = list(Path('/kaggle/input').rglob('ft_manifest.json'))
assert len(ft_manifests) == 1, ft_manifests
FT_ROOT = ft_manifests[0].parent
print('FT_ROOT =', FT_ROOT)
man = json.loads((FT_ROOT/'ft_manifest.json').read_text())
print(man['version'], man['domains'])
print(json.dumps(man['summary']['fold_sizes'], indent=1))
print('token mới theo fold:', json.dumps(man['summary']['added_tokens'], ensure_ascii=False)[:1500])


## Ghi mã vào /kaggle/working

In [ ]:
WORK_CODE = Path('/kaggle/working/il_code')
WORK_CODE.mkdir(parents=True, exist_ok=True)
(WORK_CODE/'mixed_type_attention_model.py').write_text('from __future__ import annotations\n\nfrom dataclasses import dataclass\n\nimport torch\nimport torch.nn as nn\n\n\n@dataclass(frozen=True)\nclass MixedTypeModelConfig:\n    categorical_cardinalities: list[int]\n    numerical_dim: int\n    binary_dim: int = 1\n    max_events: int = 512\n    d_model: int = 64\n    num_heads: int = 4\n    feature_layers: int = 1\n    temporal_layers: int = 2\n    ff_dim: int = 128\n    dropout: float = 0.2\n\n\nclass SelfAttentionBlock(nn.Module):\n    def __init__(\n        self,\n        *,\n        d_model: int,\n        num_heads: int,\n        ff_dim: int,\n        dropout: float,\n    ):\n        super().__init__()\n\n        if d_model % num_heads != 0:\n            raise ValueError(\n                "d_model must be divisible by num_heads"\n            )\n\n        self.norm1 = nn.LayerNorm(d_model)\n        self.attn = nn.MultiheadAttention(\n            d_model,\n            num_heads,\n            dropout=dropout,\n            batch_first=True,\n        )\n        self.attn_dropout = nn.Dropout(dropout)\n\n        self.norm2 = nn.LayerNorm(d_model)\n        self.ff = nn.Sequential(\n            nn.Linear(d_model, ff_dim),\n            nn.GELU(),\n            nn.Dropout(dropout),\n            nn.Linear(ff_dim, d_model),\n            nn.Dropout(dropout),\n        )\n\n    def forward(\n        self,\n        x: torch.Tensor,\n        *,\n        key_padding_mask: torch.Tensor | None = None,\n        return_attention: bool = False,\n    ):\n        y = self.norm1(x)\n\n        attended, weights = self.attn(\n            y,\n            y,\n            y,\n            key_padding_mask=key_padding_mask,\n            need_weights=return_attention,\n            average_attn_weights=False,\n        )\n\n        x = x + self.attn_dropout(attended)\n        x = x + self.ff(self.norm2(x))\n\n        return x, weights\n\n\nclass MixedTypeEventEncoder(nn.Module):\n    TYPE_CAT = 0\n    TYPE_NUM = 1\n    TYPE_BIN = 2\n    TYPE_DELTA = 3\n\n    def __init__(\n        self,\n        *,\n        categorical_cardinalities: list[int],\n        numerical_dim: int,\n        binary_dim: int,\n        d_model: int,\n        num_heads: int,\n        num_layers: int,\n        ff_dim: int,\n        dropout: float,\n    ):\n        super().__init__()\n\n        self.categorical_dim = len(categorical_cardinalities)\n        self.numerical_dim = int(numerical_dim)\n        self.binary_dim = int(binary_dim)\n\n        if self.binary_dim != 1:\n            raise ValueError(\n                "Current baseline expects one binary field."\n            )\n\n        self.cat_embeddings = nn.ModuleList([\n            nn.Embedding(int(cardinality), d_model)\n            for cardinality in categorical_cardinalities\n        ])\n\n        self.num_weight = nn.Parameter(\n            torch.empty(self.numerical_dim, d_model)\n        )\n        self.num_bias = nn.Parameter(\n            torch.zeros(self.numerical_dim, d_model)\n        )\n        self.num_missing_vector = nn.Parameter(\n            torch.zeros(self.numerical_dim, d_model)\n        )\n\n        self.binary_embedding = nn.Embedding(2, d_model)\n\n        self.delta_weight = nn.Parameter(\n            torch.empty(d_model)\n        )\n        self.delta_bias = nn.Parameter(\n            torch.zeros(d_model)\n        )\n\n        self.total_feature_tokens = (\n            self.categorical_dim\n            + self.numerical_dim\n            + self.binary_dim\n            + 1\n        )\n\n        self.field_embedding = nn.Embedding(\n            self.total_feature_tokens,\n            d_model,\n        )\n        self.type_embedding = nn.Embedding(4, d_model)\n\n        field_type_ids = (\n            [self.TYPE_CAT] * self.categorical_dim\n            + [self.TYPE_NUM] * self.numerical_dim\n            + [self.TYPE_BIN] * self.binary_dim\n            + [self.TYPE_DELTA]\n        )\n\n        self.register_buffer(\n            "field_type_ids",\n            torch.tensor(field_type_ids, dtype=torch.long),\n            persistent=False,\n        )\n\n        self.blocks = nn.ModuleList([\n            SelfAttentionBlock(\n                d_model=d_model,\n                num_heads=num_heads,\n                ff_dim=ff_dim,\n                dropout=dropout,\n            )\n            for _ in range(num_layers)\n        ])\n\n        self.output_norm = nn.LayerNorm(d_model)\n\n        nn.init.normal_(self.num_weight, mean=0.0, std=0.02)\n        nn.init.normal_(\n            self.num_missing_vector,\n            mean=0.0,\n            std=0.02,\n        )\n        nn.init.normal_(self.delta_weight, mean=0.0, std=0.02)\n\n    def build_feature_tokens(\n        self,\n        *,\n        x_cat: torch.Tensor,\n        x_num: torch.Tensor,\n        x_num_missing: torch.Tensor,\n        x_bin: torch.Tensor,\n        delta_t_log1p: torch.Tensor,\n    ) -> torch.Tensor:\n        cat_tokens = torch.stack(\n            [\n                emb(x_cat[:, i])\n                for i, emb in enumerate(self.cat_embeddings)\n            ],\n            dim=1,\n        )\n\n        num_tokens = (\n            x_num.unsqueeze(-1)\n            * self.num_weight.unsqueeze(0)\n            + self.num_bias.unsqueeze(0)\n        )\n\n        num_tokens = (\n            num_tokens\n            + x_num_missing.unsqueeze(-1)\n            * self.num_missing_vector.unsqueeze(0)\n        )\n\n        bin_token = self.binary_embedding(\n            x_bin[:, 0].long().clamp(0, 1)\n        ).unsqueeze(1)\n\n        delta_token = (\n            delta_t_log1p[:, 0].unsqueeze(-1)\n            * self.delta_weight.unsqueeze(0)\n            + self.delta_bias.unsqueeze(0)\n        ).unsqueeze(1)\n\n        tokens = torch.cat(\n            [\n                cat_tokens,\n                num_tokens,\n                bin_token,\n                delta_token,\n            ],\n            dim=1,\n        )\n\n        field_ids = torch.arange(\n            self.total_feature_tokens,\n            device=tokens.device,\n        )\n\n        tokens = (\n            tokens\n            + self.field_embedding(field_ids)[None, :, :]\n            + self.type_embedding(\n                self.field_type_ids\n            )[None, :, :]\n        )\n\n        return tokens\n\n    def forward(\n        self,\n        *,\n        x_cat: torch.Tensor,\n        x_num: torch.Tensor,\n        x_num_missing: torch.Tensor,\n        x_bin: torch.Tensor,\n        delta_t_log1p: torch.Tensor,\n        return_attention: bool = False,\n    ):\n        tokens = self.build_feature_tokens(\n            x_cat=x_cat,\n            x_num=x_num,\n            x_num_missing=x_num_missing,\n            x_bin=x_bin,\n            delta_t_log1p=delta_t_log1p,\n        )\n\n        last_attention = None\n\n        for i, block in enumerate(self.blocks):\n            tokens, weights = block(\n                tokens,\n                return_attention=(\n                    return_attention\n                    and i == len(self.blocks) - 1\n                ),\n            )\n            if weights is not None:\n                last_attention = weights\n\n        tokens = self.output_norm(tokens)\n\n        # Feature-level classification-style pooling.\n        event_embedding = tokens.mean(dim=1)\n\n        return event_embedding, last_attention\n\n\nclass MixedTypeSelfAttentionClassifier(nn.Module):\n    def __init__(\n        self,\n        config: MixedTypeModelConfig,\n    ):\n        super().__init__()\n        self.config = config\n\n        self.event_encoder = MixedTypeEventEncoder(\n            categorical_cardinalities=\n                config.categorical_cardinalities,\n            numerical_dim=\n                config.numerical_dim,\n            binary_dim=\n                config.binary_dim,\n            d_model=\n                config.d_model,\n            num_heads=\n                config.num_heads,\n            num_layers=\n                config.feature_layers,\n            ff_dim=\n                config.ff_dim,\n            dropout=\n                config.dropout,\n        )\n\n        self.position_embedding = nn.Embedding(\n            config.max_events,\n            config.d_model,\n        )\n\n        self.temporal_blocks = nn.ModuleList([\n            SelfAttentionBlock(\n                d_model=config.d_model,\n                num_heads=config.num_heads,\n                ff_dim=config.ff_dim,\n                dropout=config.dropout,\n            )\n            for _ in range(config.temporal_layers)\n        ])\n\n        self.temporal_norm = nn.LayerNorm(config.d_model)\n\n        self.classifier = nn.Sequential(\n            nn.Linear(config.d_model, config.d_model),\n            nn.GELU(),\n            nn.Dropout(config.dropout),\n            nn.Linear(config.d_model, 1),\n        )\n\n    def forward(\n        self,\n        *,\n        x_cat: torch.Tensor,\n        x_num: torch.Tensor,\n        x_num_missing: torch.Tensor,\n        x_bin: torch.Tensor,\n        delta_t_log1p: torch.Tensor,\n        attention_mask: torch.Tensor,\n        return_attention: bool = False,\n    ):\n        b, t, _ = x_cat.shape\n\n        if t > self.config.max_events:\n            raise ValueError(\n                f"T={t} exceeds max_events={self.config.max_events}"\n            )\n\n        valid = attention_mask.bool()\n        flat_valid = valid.reshape(-1)\n\n        flat_cat = x_cat.reshape(\n            b * t,\n            x_cat.shape[-1],\n        )[flat_valid]\n\n        flat_num = x_num.reshape(\n            b * t,\n            x_num.shape[-1],\n        )[flat_valid]\n\n        flat_missing = x_num_missing.reshape(\n            b * t,\n            x_num_missing.shape[-1],\n        )[flat_valid]\n\n        flat_bin = x_bin.reshape(\n            b * t,\n            x_bin.shape[-1],\n        )[flat_valid]\n\n        flat_delta = delta_t_log1p.reshape(\n            b * t,\n            1,\n        )[flat_valid]\n\n        event_valid, feature_attention = self.event_encoder(\n            x_cat=flat_cat,\n            x_num=flat_num,\n            x_num_missing=flat_missing,\n            x_bin=flat_bin,\n            delta_t_log1p=flat_delta,\n            return_attention=return_attention,\n        )\n\n        event_embeddings = torch.zeros(\n            b * t,\n            self.config.d_model,\n            device=x_cat.device,\n            dtype=event_valid.dtype,\n        )\n\n        event_embeddings[flat_valid] = event_valid\n\n        event_embeddings = event_embeddings.view(\n            b,\n            t,\n            self.config.d_model,\n        )\n\n        positions = torch.arange(\n            t,\n            device=x_cat.device,\n        )\n\n        temporal = (\n            event_embeddings\n            + self.position_embedding(positions)[None, :, :]\n        )\n\n        key_padding_mask = ~valid\n        last_temporal_attention = None\n\n        for i, block in enumerate(self.temporal_blocks):\n            temporal, weights = block(\n                temporal,\n                key_padding_mask=key_padding_mask,\n                return_attention=(\n                    return_attention\n                    and i == len(self.temporal_blocks) - 1\n                ),\n            )\n            if weights is not None:\n                last_temporal_attention = weights\n\n        temporal = self.temporal_norm(temporal)\n\n        mask_float = valid.unsqueeze(-1).to(temporal.dtype)\n\n        pooled = (\n            (temporal * mask_float).sum(dim=1)\n            / mask_float.sum(dim=1).clamp_min(1.0)\n        )\n\n        logits = self.classifier(pooled).squeeze(-1)\n\n        if return_attention:\n            return {\n                "logits": logits,\n                "feature_attention": feature_attention,\n                "temporal_attention": last_temporal_attention,\n                "valid_event_mask": valid,\n            }\n\n        return logits\n', encoding='utf-8')
(WORK_CODE/'s14_ft_train.py').write_text('#!/usr/bin/env python3\n"""S14b - Fine-tune mô hình B (BiLSTM hoặc Mixed-Type) trên hai miền Windows (skrghosh, FiveDirections) với từ điển mở rộng và replay OpTC (STEP 30). Chạy trên Kaggle; chạy thử CPU được.\n\nMỗi fold k:\n  1. dựng mô hình với bảng embedding mở rộng (hàng mới = bản sao hàng <UNK>); các tham số khác copy từ mô hình gốc; kiến trúc không đổi.\n  2. KIỂM TRA BẮT BUỘC: trước khi train, logit trên mẫu test của fold phải trùng dự đoán zero-shot tham chiếu (STEP 26, 29) trong dung sai; không qua thì dừng.\n  3. giai đoạn A: chỉ học hàng embedding mới + đầu phân loại; giai đoạn B: mở khóa toàn bộ, lr nhỏ. Mỗi epoch = toàn bộ mẫu TRAIN của hai miền (fold != k) + một lần rút mới từ tập replay OpTC.\n  4. đánh giá MỘT lần: mẫu test của fold (hai miền) và OpTC TEST (tuỳ chọn); ngưỡng cố định: ngưỡng VAL OpTC của mô hình gốc và 0.5. Không chọn ngưỡng, epoch hay siêu tham số theo dữ liệu mới.\nĐầu vào mô hình: 4 category (app_family, proto, resp_port_class, image_dir_class) + delta_t_log1p; numeric rỗng; x_bin = 0 (giống bản B)."""\nimport argparse\nimport importlib.util\nimport json\nimport random\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nfrom sklearn.metrics import average_precision_score, roc_auc_score\n\nCAT_IDX = [0, 1, 4, 5]  # app_family, proto, resp_port_class, image_dir_class trong x_cat 7 cột của shards\nCAT_NAMES = ["app_family", "proto", "resp_port_class", "image_dir_class"]\nEMB_PREFIX = {"bilstm": "embeddings.", "mixed": "event_encoder.cat_embeddings."}\n\n\ndef load_module(path, name):\n    sys.path.insert(0, str(Path(path).parent))  # 05b_train_bilstm.py import network_dl_dataset_v1_1 cùng thư mục\n    spec = importlib.util.spec_from_file_location(name, path)\n    mod = importlib.util.module_from_spec(spec)\n    sys.modules[name] = mod\n    spec.loader.exec_module(mod)\n    return mod\n\n\n# ---------------------------------------------------------------- dữ liệu\ndef read_shard(path):\n    with np.load(path, allow_pickle=False) as z:\n        return {k: z[k] for k in ("x_cat", "delta_t_log1p", "sample_offsets")}\n\n\ndef sample_from(arrs, i, label, sid):\n    off = arrs["sample_offsets"]\n    a, b = int(off[i]), int(off[i + 1])\n    return {"x_cat": arrs["x_cat"][a:b][:, CAT_IDX].astype(np.int64), "delta": arrs["delta_t_log1p"][a:b].astype(np.float32), "label": int(label), "sample_id": sid}\n\n\ndef materialize(rows):\n    """rows: DataFrame có cột root, shard_file, shard_sample_index, label, sample_id -> list sample (giữ thứ tự rows)."""\n    out = [None] * len(rows)\n    rows = rows.reset_index(drop=True)\n    for (root, shard), g in rows.groupby(["root", "shard_file"], sort=False):\n        arrs = read_shard(Path(root) / shard)\n        for pos, r in zip(g.index, g.itertuples()):\n            out[pos] = sample_from(arrs, int(r.shard_sample_index), r.label, r.sample_id)\n    return out\n\n\ndef collate(batch):\n    lengths = torch.tensor([len(s["delta"]) for s in batch], dtype=torch.long)\n    T, B = int(lengths.max()), len(batch)\n    x_cat = torch.zeros(B, T, len(CAT_IDX), dtype=torch.long)\n    delta = torch.zeros(B, T, 1)\n    mask = torch.zeros(B, T, dtype=torch.bool)\n    for i, s in enumerate(batch):\n        t = len(s["delta"])\n        x_cat[i, :t] = torch.from_numpy(s["x_cat"])\n        delta[i, :t, 0] = torch.from_numpy(s["delta"])\n        mask[i, :t] = True\n    return {"x_cat": x_cat, "x_num": torch.zeros(B, T, 0), "x_num_missing": torch.zeros(B, T, 0), "x_bin": torch.zeros(B, T, 1), "delta_t_log1p": delta, "mask": mask,\n            "lengths": lengths, "labels": torch.tensor([s["label"] for s in batch], dtype=torch.float32), "sample_ids": [s["sample_id"] for s in batch]}\n\n\ndef forward(model, kind, b, device):\n    kw = {k: b[k].to(device) for k in ("x_cat", "x_num", "x_num_missing", "x_bin", "delta_t_log1p")}\n    if kind == "bilstm":\n        return model(lengths=b["lengths"], **kw)\n    return model(attention_mask=b["mask"].to(device), **kw)\n\n\n@torch.no_grad()\ndef predict(model, kind, samples, device, bs):\n    model.eval()\n    logit, labels, ids = [], [], []\n    for i in range(0, len(samples), bs):\n        b = collate(samples[i:i + bs])\n        logit.append(forward(model, kind, b, device).float().cpu().numpy())\n        labels.append(b["labels"].numpy())\n        ids += b["sample_ids"]\n    logit = np.concatenate(logit) if logit else np.zeros(0)\n    prob = torch.sigmoid(torch.from_numpy(logit).half()).float().numpy()  # giống mô phỏng fp16 ở STEP 26\n    return pd.DataFrame({"sample_id": ids, "label": (np.concatenate(labels) if labels else np.zeros(0)).astype(int), "logit": logit, "probability": prob})\n\n\ndef metrics_at(y, p, thr):\n    pred = (p >= thr).astype(int)\n    tp = int(((y == 1) & (pred == 1)).sum()); fp = int(((y == 0) & (pred == 1)).sum()); fn = int(((y == 1) & (pred == 0)).sum()); tn = int(((y == 0) & (pred == 0)).sum())\n    pr = tp / (tp + fp) if tp + fp else 0.0\n    rc = tp / (tp + fn) if tp + fn else 0.0\n    f1 = 2 * pr * rc / (pr + rc) if pr + rc else 0.0\n    return {"threshold": float(thr), "tp": tp, "fp": fp, "fn": fn, "tn": tn, "precision": round(pr, 4), "recall": round(rc, 4), "f1": round(f1, 4)}\n\n\ndef summarize(df, thresholds):\n    y, p, lg = df.label.to_numpy(), df.probability.to_numpy(), df.logit.to_numpy()\n    out = {"n": int(len(df)), "positives": int(y.sum()), "at": {name: metrics_at(y, p, t) for name, t in thresholds.items()}}\n    if 0 < y.sum() < len(y):\n        out["pr_auc"] = round(float(average_precision_score(y, lg)), 4)\n        out["roc_auc"] = round(float(roc_auc_score(y, lg)), 4)\n    return out\n\n\n# ---------------------------------------------------------------- mô hình\ndef build_model(kind, cfg, cards, code_dir):\n    if kind == "bilstm":\n        m = load_module(Path(code_dir) / "05b_train_bilstm.py", "train_bilstm")\n        base_dims = [m.embedding_dim(c) for c in cfg["categorical_cardinalities"]]  # giữ nguyên chiều embedding của mô hình gốc dù số hàng tăng (hàm gốc suy chiều từ số hàng)\n        it = iter(base_dims)\n        m.embedding_dim = lambda c: next(it)\n        return m.BiLSTMBaseline(categorical_cardinalities=cards, numerical_dim=0, event_dim=cfg["event_dim"], hidden_dim=cfg["hidden_dim"], lstm_layers=cfg["lstm_layers"], dropout=cfg["dropout"])\n    m = load_module(Path(code_dir) / "mixed_type_attention_model.py", "mixed_model")\n    return m.MixedTypeSelfAttentionClassifier(m.MixedTypeModelConfig(categorical_cardinalities=cards, numerical_dim=0, binary_dim=1, max_events=cfg["max_events"], d_model=cfg["d_model"],\n                                                                      num_heads=cfg["num_heads"], feature_layers=cfg["feature_layers"], temporal_layers=cfg["temporal_layers"], ff_dim=cfg["ff_dim"],\n                                                                      dropout=cfg["dropout"]))\n\n\ndef transfer_weights(model, base_sd, old_cards, kind):\n    """Copy tham số gốc; hàng embedding mới = bản sao hàng <UNK> (id 0). Trả về {tên tham số embedding: số hàng cũ}."""\n    sd = model.state_dict()\n    new_sd, old_rows = {}, {}\n    for key, tensor in sd.items():\n        if key not in base_sd:\n            raise KeyError(f"tham số {key} không có trong mô hình gốc")\n        base = base_sd[key]\n        if key.startswith(EMB_PREFIX[kind]) and key.endswith(".weight"):\n            idx = int(key[len(EMB_PREFIX[kind]):].split(".")[0])\n            assert base.shape[1] == tensor.shape[1], f"chiều embedding đổi ở {key}: {tuple(base.shape)} -> {tuple(tensor.shape)}"\n            assert base.shape[0] == old_cards[idx] and tensor.shape[0] >= base.shape[0]\n            w = tensor.clone()\n            w[:base.shape[0]] = base\n            w[base.shape[0]:] = base[0:1]\n            new_sd[key] = w\n            old_rows[key] = base.shape[0]\n        else:\n            assert base.shape == tensor.shape, (key, base.shape, tensor.shape)\n            new_sd[key] = base\n    model.load_state_dict(new_sd)\n    return old_rows\n\n\ndef set_stage(model, kind, stage, old_rows):\n    """A: chỉ hàng embedding mới và đầu phân loại. B: tất cả. Trả về danh sách hook cần gỡ."""\n    handles = []\n    for name, p in model.named_parameters():\n        if stage == "B":\n            p.requires_grad = True\n        elif name in old_rows:\n            p.requires_grad = True\n            mask = torch.zeros(p.shape[0], 1, device=p.device)\n            mask[old_rows[name]:] = 1.0\n            handles.append(p.register_hook(lambda g, m=mask: g * m))\n        else:\n            p.requires_grad = name.startswith("classifier.")\n    return handles\n\n\ndef seed_everything(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\n# ---------------------------------------------------------------- chạy một fold\ndef run_fold(args, k, ctx):\n    dev = ctx["device"]\n    kind = args.model\n    ft = Path(args.ft_root)\n    fold_dir = ft / f"fold{k}"\n    spec = json.load(open(fold_dir / "spec_ext.json", encoding="utf-8"))\n    cards = [max(e["id"] for e in spec["categorical"]["fields"][f]["vocab"]) + 1 for f in CAT_NAMES]\n    cfg, old_cards = ctx["cfg"], ctx["cfg"]["categorical_cardinalities"]\n    print(f"[fold {k}] cardinalities cũ {old_cards} -> mở rộng {cards}", flush=True)\n    seed_everything(args.seed + k)\n    model = build_model(kind, cfg, cards, args.code_dir)\n    old_rows = transfer_weights(model, ctx["base_sd"], old_cards, kind)\n    model.to(dev)\n    # dữ liệu hai miền\n    folds = pd.read_csv(ft / "folds.csv")\n    rows = []\n    for ds in ctx["domains"]:\n        sm = pd.read_csv(fold_dir / ds / "data/samples.csv")[["sample_id", "shard_file", "shard_sample_index"]]\n        m = sm.merge(folds[folds.dataset == ds][["sample_id", "fold", "label"]], on="sample_id")\n        m["root"] = str(fold_dir / ds)\n        m["dataset"] = ds\n        rows.append(m)\n    rows = pd.concat(rows, ignore_index=True)\n    samples = materialize(rows)\n    tr_idx = [i for i in range(len(rows)) if rows.fold.iat[i] != k]\n    te_idx = [i for i in range(len(rows)) if rows.fold.iat[i] == k]\n    train_new = [samples[i] for i in tr_idx]\n    test_samples = [samples[i] for i in te_idx]\n    print(f"[fold {k}] train {len(train_new)} mẫu (dương {sum(s[\'label\'] for s in train_new)}), test {len(test_samples)} (dương {sum(s[\'label\'] for s in test_samples)})", flush=True)\n    thr = {"optc_val_threshold": ctx["thr"], "0.5": 0.5}\n    # kiểm tra khớp zero-shot\n    pre = predict(model, kind, test_samples, dev, args.eval_batch)\n    pre["dataset"] = rows.dataset.iloc[te_idx].to_numpy()\n    worst = 0.0\n    for ds in ctx["domains"]:\n        ref = pd.read_csv(ft / "ref_zeroshot" / f"predictions_{ds}_{kind}.csv")[["sample_id", "logit"]].rename(columns={"logit": "ref_logit"})\n        m = pre[pre.dataset == ds].merge(ref, on="sample_id")\n        assert len(m) == (pre.dataset == ds).sum(), f"thiếu mẫu tham chiếu {ds}"\n        d = float((m.logit - m.ref_logit).abs().max())\n        worst = max(worst, d)\n        print(f"[fold {k}] khớp zero-shot {ds}: chênh logit lớn nhất {d:.5f} trên {len(m)} mẫu", flush=True)\n    if worst > args.equiv_tol:\n        raise RuntimeError(f"mô hình mở rộng chưa train KHÔNG khớp zero-shot (chênh {worst:.5f} > {args.equiv_tol}); dừng")\n    result = {"fold": k, "cardinalities_old": old_cards, "cardinalities_ext": cards, "equivalence_max_logit_diff": worst,\n              "zero_shot_on_test": summarize(pre, thr), "added_tokens": spec["extension"]["added"], "history": []}\n    # replay\n    rep = ctx["replay"]\n    rep_pos = [s for s in rep if s["label"] == 1]\n    rep_neg = [s for s in rep if s["label"] == 0]\n    n_pos = sum(s["label"] for s in train_new) + args.replay_pos\n    n_neg = len(train_new) - sum(s["label"] for s in train_new) + (args.replay_per_epoch - args.replay_pos)\n    pos_weight = torch.tensor(n_neg / max(n_pos, 1), dtype=torch.float32, device=dev)\n    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)\n    amp = dev.type == "cuda" and not args.no_amp\n    scaler = torch.amp.GradScaler("cuda", enabled=amp)\n    rng = np.random.default_rng(args.seed * 1000 + k)\n    ep_global = 0\n    for stage, epochs, lr, wd in (("A", args.epochs_a, args.lr_a, 0.0), ("B", args.epochs_b, args.lr_b_bilstm if kind == "bilstm" else args.lr_b_mixed, args.weight_decay)):\n        handles = set_stage(model, kind, stage, old_rows)\n        params = [p for p in model.parameters() if p.requires_grad]\n        opt = torch.optim.AdamW(params, lr=lr, weight_decay=wd)\n        print(f"[fold {k}] giai đoạn {stage}: {epochs} epoch, lr {lr}, tham số học {sum(p.numel() for p in params):,}", flush=True)\n        for ep in range(epochs):\n            ep_global += 1\n            draw = [rep_pos[i] for i in rng.permutation(len(rep_pos))[:args.replay_pos]] + [rep_neg[i] for i in rng.permutation(len(rep_neg))[:args.replay_per_epoch - args.replay_pos]]\n            epoch_samples = train_new + draw\n            order = rng.permutation(len(epoch_samples))\n            model.train()\n            tot, nb, t0 = 0.0, 0, time.time()\n            for i in range(0, len(order), args.batch_size):\n                b = collate([epoch_samples[j] for j in order[i:i + args.batch_size]])\n                opt.zero_grad(set_to_none=True)\n                with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=amp):\n                    loss = crit(forward(model, kind, b, dev).float(), b["labels"].to(dev))\n                scaler.scale(loss).backward()\n                scaler.unscale_(opt)\n                torch.nn.utils.clip_grad_norm_(params, args.grad_clip)\n                scaler.step(opt)\n                scaler.update()\n                tot += float(loss.item())\n                nb += 1\n            result["history"].append({"stage": stage, "epoch": ep_global, "train_loss": round(tot / max(nb, 1), 5), "seconds": round(time.time() - t0, 1)})\n            print(f"[fold {k}] {stage} epoch {ep_global}: loss {tot / max(nb, 1):.5f} ({time.time() - t0:.0f}s)", flush=True)\n        for h in handles:\n            h.remove()\n    # đánh giá một lần\n    post = predict(model, kind, test_samples, dev, args.eval_batch)\n    post["dataset"] = rows.dataset.iloc[te_idx].to_numpy()\n    post["fold"] = k\n    result["after_on_test"] = summarize(post, thr)\n    result["after_on_test_by_dataset"] = {ds: summarize(post[post.dataset == ds], thr) for ds in ctx["domains"]}\n    result["before_on_test_by_dataset"] = {ds: summarize(pre[pre.dataset == ds], thr) for ds in ctx["domains"]}\n    out = Path(args.out_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    post.to_csv(out / f"predictions_{kind}_fold{k}_newdomains.csv", index=False)\n    torch.save({"model_state_dict": model.state_dict(), "cardinalities": cards, "fold": k}, out / f"model_{kind}_fold{k}.pt")\n    if args.optc_eval != "none":\n        result["optc_test"] = eval_optc(model, kind, args, ctx, thr, k)\n    json.dump(result, open(out / f"result_{kind}_fold{k}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)\n    print(f"[fold {k}] xong; test sau fine-tune:", json.dumps(result["after_on_test"]["at"], ensure_ascii=False), flush=True)\n    return result\n\n\ndef eval_optc(model, kind, args, ctx, thr, k):\n    """OpTC TEST (đọc streaming theo shard); với mô hình mở rộng id cũ < 61 nên dùng nguyên shards OpTC."""\n    t0 = time.time()\n    man = ctx["optc_manifest"]\n    te = man[man.split == "test"]\n    if args.max_optc_test:\n        te = pd.concat([te[te.label == 1], te[te.label == 0].head(args.max_optc_test)])\n    root = Path(args.optc_root)\n    frames = []\n    for shard, g in te.groupby("shard_file", sort=False):\n        arrs = read_shard(root / shard)\n        samples = [sample_from(arrs, int(r.shard_sample_index), r.label, r.sample_id) for r in g.itertuples()]\n        frames.append(predict(model, kind, samples, ctx["device"], args.eval_batch))\n    df = pd.concat(frames, ignore_index=True)\n    df.to_csv(Path(args.out_dir) / f"predictions_{kind}_fold{k}_optc_test.csv", index=False)\n    res = summarize(df, thr)\n    res["seconds"] = round(time.time() - t0)\n    print(f"[fold {k}] OpTC TEST: {json.dumps(res[\'at\'], ensure_ascii=False)} ({res[\'seconds\']}s)", flush=True)\n    return res\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--model", choices=["bilstm", "mixed"], required=True)\n    ap.add_argument("--ft-root", required=True, help="gói v3_ft_kaggle_upload (có ft_manifest.json)")\n    ap.add_argument("--optc-root", required=True, help="gốc dataset OpTC v2 (split/, shards/)")\n    ap.add_argument("--code-dir", required=True, help="thư mục chứa 05b_train_bilstm.py và mixed_type_attention_model.py")\n    ap.add_argument("--out-dir", required=True)\n    ap.add_argument("--folds", default="0,1,2")\n    ap.add_argument("--seed", type=int, default=42)\n    ap.add_argument("--epochs-a", type=int, default=3)\n    ap.add_argument("--epochs-b", type=int, default=3)\n    ap.add_argument("--lr-a", type=float, default=1e-3)\n    ap.add_argument("--lr-b-bilstm", type=float, default=1e-4)\n    ap.add_argument("--lr-b-mixed", type=float, default=3e-5)\n    ap.add_argument("--weight-decay", type=float, default=1e-4)\n    ap.add_argument("--grad-clip", type=float, default=1.0)\n    ap.add_argument("--replay-per-epoch", type=int, default=600)\n    ap.add_argument("--replay-pos", type=int, default=150)\n    ap.add_argument("--batch-size", type=int, default=0, help="0 = mặc định theo mô hình (BiLSTM 32, Mixed 16)")\n    ap.add_argument("--eval-batch", type=int, default=0)\n    ap.add_argument("--equiv-tol", type=float, default=0.02)\n    ap.add_argument("--optc-eval", choices=["test", "none"], default="test")\n    ap.add_argument("--max-optc-test", type=int, default=0, help="chỉ để chạy thử: giới hạn số mẫu âm OpTC TEST")\n    ap.add_argument("--no-amp", action="store_true")\n    ap.add_argument("--cpu", action="store_true")\n    args = ap.parse_args()\n    args.batch_size = args.batch_size or (32 if args.model == "bilstm" else 16)\n    args.eval_batch = args.eval_batch or (128 if args.model == "bilstm" else 32)\n    dev = torch.device("cpu" if args.cpu or not torch.cuda.is_available() else "cuda")\n    ft = Path(args.ft_root)\n    man = json.load(open(ft / "ft_manifest.json", encoding="utf-8"))\n    base = ft / "base_models" / args.model\n    cfg = json.load(open(base / "config.json", encoding="utf-8"))\n    thr = float(json.load(open(base / "threshold.json", encoding="utf-8"))["threshold"])\n    ck = torch.load(base / "best_model.pt", map_location="cpu", weights_only=False)\n    print(f"[init] mô hình {args.model}, thiết bị {dev}, ngưỡng VAL OpTC {thr}, thư mục ra {args.out_dir}", flush=True)\n    optc_manifest = pd.read_csv(Path(args.optc_root) / "split/network_sample_manifest.csv")\n    # tập replay (đọc một lần)\n    rr = pd.read_csv(ft / "replay_R.csv")\n    rr["root"] = str(Path(args.optc_root))\n    t0 = time.time()\n    replay = materialize(rr)\n    print(f"[init] replay R: {len(replay)} mẫu (dương {sum(s[\'label\'] for s in replay)}) nạp trong {time.time() - t0:.0f}s", flush=True)\n    ctx = {"device": dev, "cfg": cfg, "base_sd": ck["model_state_dict"], "thr": thr, "domains": man["domains"], "replay": replay, "optc_manifest": optc_manifest}\n    results = []\n    for k in [int(x) for x in args.folds.split(",")]:\n        results.append(run_fold(args, k, ctx))\n    json.dump({"model": args.model, "args": vars(args), "folds": [r["fold"] for r in results]}, open(Path(args.out_dir) / f"run_{args.model}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)\n    print("[xong]", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
(WORK_CODE/'s36_il_train.py').write_text('#!/usr/bin/env python3\n"""S36c - Fine-tune / Incremental Learning OpTC -> ATLASv2 (STEP 48), mo hinh B (BiLSTM hoac Mixed-Type). Chay tren Kaggle; chay thu CPU duoc.\n\nKhac s14_ft_train.py (STEP 30):\n  - co VAL (cua so vb == 0 trong cac fold con lai) de chon checkpoint (VAL PR-AUC + 0.1 * F1 tot nhat) va nguong (F1 toi da tren VAL, giua cao nguyen). TEST chi tinh MOT lan.\n  - mo dan (gradual unfreezing) theo STEP 44: A (hang embedding moi + dau phan loai) -> B1 (+ encoder su kien) -> B2 (+ tang chuoi) -> C (+ embedding cu, lr nho);\n    moi giai doan dung toi da N epoch, dung som theo VAL, nap lai checkpoint tot nhat cua giai doan truoc khi mo tiep.\n  - replay OpTC that (STEP 30) tron vao moi epoch; khong so sanh "truoc fine-tune"; chi bao ma tran nham lan TRAIN, VAL, TEST (ATLASv2) va OpTC TEST.\n  - kiem tra tuong duong khoi tao: mo hinh mo rong (chua train) cho logit giong mo hinh goc khi id moi duoc doi thanh <UNK>.\nTai su dung ham tu s14_ft_train.py (cung thu muc)."""\nimport argparse\nimport json\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nfrom sklearn.metrics import average_precision_score, roc_auc_score\n\nsys.path.insert(0, str(Path(__file__).parent))\nimport s14_ft_train as T  # noqa: E402\n\n\n# nguong goc 1.0 (Mixed-Type) la nguong tren xac suat fp16 (sigmoid lam tron thanh 1.0 khi logit > ~8.3): doi sang logit bang cach chan 1 - 2^-12\n\n\ndef sigmoid(x):\n    return 1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))\n\n\ndef group_of(name):\n    if name.startswith("classifier."):\n        return "head"\n    if name.startswith("embeddings.") or name.startswith("event_encoder.cat_embeddings."):\n        return "emb"\n    if name.startswith("event_encoder."):\n        return "enc"\n    return "seq"  # lstm.*, temporal_blocks.*, temporal_norm.*, position_embedding.*\n\n\ndef conf_at(y, lg, thr_logit):\n    pred = (lg >= thr_logit).astype(int)\n    tp = int(((y == 1) & (pred == 1)).sum()); fp = int(((y == 0) & (pred == 1)).sum())\n    fn = int(((y == 1) & (pred == 0)).sum()); tn = int(((y == 0) & (pred == 0)).sum())\n    pr = tp / (tp + fp) if tp + fp else 0.0\n    rc = tp / (tp + fn) if tp + fn else 0.0\n    f1 = 2 * pr * rc / (pr + rc) if pr + rc else 0.0\n    return {"threshold_logit": round(float(thr_logit), 5), "threshold_prob": round(float(sigmoid(thr_logit)), 6), "tp": tp, "fp": fp, "fn": fn, "tn": tn,\n            "precision": round(pr, 4), "recall": round(rc, 4), "f1": round(f1, 4)}\n\n\ndef best_threshold(y, lg):\n    """F1 toi da tren VAL; neu co cao nguyen (nhieu nguong cho F1 bang nhau) lay diem giua cao nguyen (theo logit)."""\n    u = np.unique(lg)\n    if len(u) == 1:\n        return float(u[0]) + 1e-6\n    cand = np.r_[u[0] - 1.0, (u[:-1] + u[1:]) / 2.0, u[-1] + 1.0]\n    f1 = np.array([conf_at(y, lg, t)["f1"] for t in cand])\n    top = np.flatnonzero(f1 >= f1.max() - 1e-12)\n    runs = np.split(top, np.flatnonzero(np.diff(top) > 1) + 1)\n    run = max(runs, key=len)\n    return float((cand[run[0]] + cand[run[-1]]) / 2.0)\n\n\ndef summarize(df, thr_logit):\n    y, lg = df.label.to_numpy(), df.logit.to_numpy()\n    out = {"n": int(len(df)), "positives": int(y.sum()), "confusion": conf_at(y, lg, thr_logit)}\n    if 0 < y.sum() < len(y):\n        out["pr_auc"] = round(float(average_precision_score(y, lg)), 4)\n        out["roc_auc"] = round(float(roc_auc_score(y, lg)), 4)\n    return out\n\n\ndef equivalence_check(model_ext, base_sd, cfg, old_cards, kind, samples, dev, code_dir, bs):\n    base = T.build_model(kind, cfg, old_cards, code_dir)\n    base.load_state_dict(base_sd)\n    base.to(dev)\n    mapped = []\n    for s in samples:\n        x = s["x_cat"].copy()\n        for j, c in enumerate(old_cards):\n            x[:, j] = np.where(x[:, j] >= c, 0, x[:, j])\n        mapped.append({**s, "x_cat": x})\n    a = T.predict(model_ext, kind, samples, dev, bs).logit.to_numpy()\n    b = T.predict(base, kind, mapped, dev, bs).logit.to_numpy()\n    return float(np.abs(a - b).max())\n\n\ndef stage_plan(args):\n    lb = args.lr_b_bilstm if args.model == "bilstm" else args.lr_b_mixed\n    return [  # (ten, so epoch toi da, lr co so, nhom hoat dong, he so lr theo nhom)\n        ("A", args.epochs_a, args.lr_a, ("head", "emb"), {"head": 1.0, "emb": 1.0}),\n        ("B1", args.epochs_b, lb, ("head", "emb", "enc"), {"head": 1.0, "emb": 1.0, "enc": 1.0}),\n        ("B2", args.epochs_b, lb, ("head", "emb", "enc", "seq"), {"head": 1.0, "emb": 1.0, "enc": 1.0, "seq": 0.5}),\n        ("C", args.epochs_b, lb, ("head", "emb", "enc", "seq"), {"head": 0.5, "emb": 0.1, "enc": 0.5, "seq": 0.25}),\n    ]\n\n\ndef set_requires_grad(model, active, mask_old_emb, old_rows):\n    handles = []\n    for name, p in model.named_parameters():\n        g = group_of(name)\n        p.requires_grad = g in active\n        if g == "emb" and p.requires_grad and mask_old_emb:\n            mask = torch.zeros(p.shape[0], 1, device=p.device)\n            mask[old_rows[name]:] = 1.0\n            handles.append(p.register_hook(lambda gr, m=mask: gr * m))\n    return handles\n\n\ndef run_fold(args, k, ctx):\n    dev = ctx["device"]\n    kind = args.model\n    ft = Path(args.ft_root)\n    fold_dir = ft / f"fold{k}"\n    spec = json.load(open(fold_dir / "spec_ext.json", encoding="utf-8"))\n    cards = [max(e["id"] for e in spec["categorical"]["fields"][f]["vocab"]) + 1 for f in T.CAT_NAMES]\n    cfg, old_cards = ctx["cfg"], ctx["cfg"]["categorical_cardinalities"]\n    print(f"[fold {k}] cardinalities cu {old_cards} -> mo rong {cards}", flush=True)\n    T.seed_everything(args.seed + k)\n    model = T.build_model(kind, cfg, cards, args.code_dir)\n    old_rows = T.transfer_weights(model, ctx["base_sd"], old_cards, kind)\n    model.to(dev)\n    folds = pd.read_csv(ft / "folds.csv")\n    sm = pd.read_csv(fold_dir / "atlasv2/data/samples.csv")[["sample_id", "shard_file", "shard_sample_index"]]\n    rows = sm.merge(folds[["sample_id", "fold", "vb", "label", "scenario", "kind", "hostname"]], on="sample_id")\n    assert len(rows) == len(sm) == len(folds)\n    rows["root"] = str(fold_dir / "atlasv2")\n    rows["role"] = np.where(rows.fold == k, "test", np.where(rows.vb == 0, "val", "train"))\n    samples = T.materialize(rows)\n    sel = {r: [i for i in range(len(rows)) if rows.role.iat[i] == r] for r in ("train", "val", "test")}\n    train_new = [samples[i] for i in sel["train"]]\n    val_s = [samples[i] for i in sel["val"]]\n    test_s = [samples[i] for i in sel["test"]]\n    npos = lambda L: sum(s["label"] for s in L)\n    print(f"[fold {k}] train {len(train_new)} (duong {npos(train_new)}) | val {len(val_s)} (duong {npos(val_s)}) | test {len(test_s)} (duong {npos(test_s)})", flush=True)\n    worst = equivalence_check(model, ctx["base_sd"], cfg, old_cards, kind, test_s[:200], dev, args.code_dir, args.eval_batch)\n    print(f"[fold {k}] kiem tra tuong duong khoi tao: chenh logit lon nhat {worst:.6f}", flush=True)\n    if worst > args.equiv_tol:\n        raise RuntimeError(f"mo hinh mo rong chua train KHONG tuong duong mo hinh goc (chenh {worst:.5f} > {args.equiv_tol})")\n    result = {"fold": k, "cardinalities_old": old_cards, "cardinalities_ext": cards, "equivalence_max_logit_diff": worst, "added_tokens": spec["extension"]["added"],\n              "sizes": {r: {"n": len(sel[r]), "positives": npos([samples[i] for i in sel[r]])} for r in sel}, "history": []}\n    rep = ctx["replay"]\n    rep_pos = [s for s in rep if s["label"] == 1]\n    rep_neg = [s for s in rep if s["label"] == 0]\n    n_pos = npos(train_new) + args.replay_pos\n    n_neg = len(train_new) - npos(train_new) + (args.replay_per_epoch - args.replay_pos)\n    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(n_neg / max(n_pos, 1), dtype=torch.float32, device=dev))\n    amp = dev.type == "cuda" and not args.no_amp\n    scaler = torch.amp.GradScaler("cuda", enabled=amp)\n    rng = np.random.default_rng(args.seed * 1000 + k)\n    best = {"score": -1.0, "state": None, "stage": None, "epoch": 0}\n    ep_global = 0\n    for stage, max_ep, lr, active, mult in stage_plan(args):\n        handles = set_requires_grad(model, active, mask_old_emb=(stage != "C"), old_rows=old_rows)\n        groups = []\n        for g in active:\n            ps = [p for n, p in model.named_parameters() if group_of(n) == g and p.requires_grad]\n            if ps:\n                groups.append({"params": ps, "lr": lr * mult[g], "weight_decay": 0.0 if g == "emb" else args.weight_decay})\n        opt = torch.optim.AdamW(groups)\n        ntr = sum(p.numel() for gp in groups for p in gp["params"])\n        print(f"[fold {k}] giai doan {stage}: toi da {max_ep} epoch, lr co so {lr}, nhom {list(active)}, tham so co the hoc {ntr:,}", flush=True)\n        stage_best, bad_ep = -1.0, 0\n        for ep in range(max_ep):\n            ep_global += 1\n            draw = [rep_pos[i] for i in rng.permutation(len(rep_pos))[:args.replay_pos]] + [rep_neg[i] for i in rng.permutation(len(rep_neg))[:args.replay_per_epoch - args.replay_pos]]\n            epoch_samples = train_new + draw\n            order = rng.permutation(len(epoch_samples))\n            model.train()\n            tot, nb, t0 = 0.0, 0, time.time()\n            params = [p for gp in groups for p in gp["params"]]\n            for i in range(0, len(order), args.batch_size):\n                b = T.collate([epoch_samples[j] for j in order[i:i + args.batch_size]])\n                opt.zero_grad(set_to_none=True)\n                with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=amp):\n                    loss = crit(T.forward(model, kind, b, dev).float(), b["labels"].to(dev))\n                scaler.scale(loss).backward()\n                scaler.unscale_(opt)\n                torch.nn.utils.clip_grad_norm_(params, args.grad_clip)\n                scaler.step(opt)\n                scaler.update()\n                tot += float(loss.item())\n                nb += 1\n            vdf = T.predict(model, kind, val_s, dev, args.eval_batch)\n            vy, vl = vdf.label.to_numpy(), vdf.logit.to_numpy()\n            vpr = float(average_precision_score(vy, vl)) if 0 < vy.sum() < len(vy) else 0.0\n            vthr = best_threshold(vy, vl)\n            vf1 = conf_at(vy, vl, vthr)["f1"]\n            score = vpr + 0.1 * vf1\n            result["history"].append({"stage": stage, "epoch": ep_global, "train_loss": round(tot / max(nb, 1), 5), "val_pr_auc": round(vpr, 4), "val_f1": vf1, "val_score": round(score, 4),\n                                      "seconds": round(time.time() - t0, 1)})\n            print(f"[fold {k}] {stage} epoch {ep_global}: loss {tot / max(nb, 1):.5f} | VAL PR-AUC {vpr:.4f} F1 {vf1:.4f} ({time.time() - t0:.0f}s)", flush=True)\n            if score > best["score"] + 1e-6:\n                best = {"score": score, "state": {n: v.detach().cpu().clone() for n, v in model.state_dict().items()}, "stage": stage, "epoch": ep_global}\n            if score > stage_best + 1e-6:\n                stage_best, bad_ep = score, 0\n            else:\n                bad_ep += 1\n                if bad_ep >= args.patience:\n                    print(f"[fold {k}] giai doan {stage}: dung som (khong cai thien VAL {args.patience} epoch)", flush=True)\n                    break\n        for h in handles:\n            h.remove()\n        model.load_state_dict(best["state"])  # mo tiep tu checkpoint tot nhat tren VAL\n    print(f"[fold {k}] checkpoint tot nhat: giai doan {best[\'stage\']}, epoch {best[\'epoch\']}, VAL score {best[\'score\']:.4f}", flush=True)\n    model.load_state_dict(best["state"])\n    # ngưỡng chọn trên VAL, đánh giá một lần\n    vdf = T.predict(model, kind, val_s, dev, args.eval_batch)\n    thr_logit = best_threshold(vdf.label.to_numpy(), vdf.logit.to_numpy())\n    out = Path(args.out_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    allp = []\n    for role in ("train", "val", "test"):\n        df = T.predict(model, kind, [samples[i] for i in sel[role]], dev, args.eval_batch)\n        meta = rows.iloc[sel[role]][["scenario", "kind", "hostname"]].reset_index(drop=True)\n        df = pd.concat([df, meta], axis=1)\n        df["role"] = role\n        df["fold"] = k\n        allp.append(df)\n        result[f"atlasv2_{role}"] = summarize(df, thr_logit)\n    pd.concat(allp).to_csv(out / f"predictions_{kind}_fold{k}_atlasv2.csv", index=False)\n    result.update({"selected_on": "VAL (ATLASv2)", "threshold_logit": thr_logit, "threshold_prob": float(sigmoid(thr_logit)), "best_stage": best["stage"], "best_epoch": best["epoch"]})\n    te = allp[2]\n    result["atlasv2_test_by_scenario"] = {sc: {"n": int(len(g)), "positives": int(g.label.sum()), "predicted_positive": int((g.logit >= thr_logit).sum()),\n                                               "tp": int(((g.label == 1) & (g.logit >= thr_logit)).sum())} for sc, g in te.groupby("scenario")}\n    torch.save({"model_state_dict": model.state_dict(), "cardinalities": cards, "fold": k, "threshold_logit": thr_logit}, out / f"model_{kind}_fold{k}.pt")\n    if args.optc_eval != "none":\n        res = T.eval_optc(model, kind, args, ctx, {"x": 0.5}, k)\n        df = pd.read_csv(out / f"predictions_{kind}_fold{k}_optc_test.csv")\n        result["optc_test"] = {"n": int(len(df)), "positives": int(df.label.sum()), "at_val_threshold": conf_at(df.label.to_numpy(), df.logit.to_numpy(), thr_logit),\n                               "at_optc_base_threshold": conf_at(df.label.to_numpy(), df.logit.to_numpy(), float(np.log(min(ctx["thr"], 1 - 2.0 ** -12) / (1 - min(ctx["thr"], 1 - 2.0 ** -12))))),\n                               "pr_auc": round(float(average_precision_score(df.label, df.logit)), 4), "roc_auc": round(float(roc_auc_score(df.label, df.logit)), 4)}\n    json.dump(result, open(out / f"result_{kind}_fold{k}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)\n    print(f"[fold {k}] xong. TEST ATLASv2:", json.dumps(result["atlasv2_test"]["confusion"], ensure_ascii=False), flush=True)\n    return result\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--model", choices=["bilstm", "mixed"], required=True)\n    ap.add_argument("--ft-root", required=True)\n    ap.add_argument("--optc-root", required=True)\n    ap.add_argument("--code-dir", required=True)\n    ap.add_argument("--out-dir", required=True)\n    ap.add_argument("--folds", default="0,1,2")\n    ap.add_argument("--seed", type=int, default=42)\n    ap.add_argument("--epochs-a", type=int, default=8)\n    ap.add_argument("--epochs-b", type=int, default=10)\n    ap.add_argument("--patience", type=int, default=3)\n    ap.add_argument("--lr-a", type=float, default=1e-3)\n    ap.add_argument("--lr-b-bilstm", type=float, default=3e-4)\n    ap.add_argument("--lr-b-mixed", type=float, default=1e-4)\n    ap.add_argument("--weight-decay", type=float, default=1e-4)\n    ap.add_argument("--grad-clip", type=float, default=1.0)\n    ap.add_argument("--replay-per-epoch", type=int, default=600)\n    ap.add_argument("--replay-pos", type=int, default=150)\n    ap.add_argument("--batch-size", type=int, default=0)\n    ap.add_argument("--eval-batch", type=int, default=0)\n    ap.add_argument("--equiv-tol", type=float, default=0.02)\n    ap.add_argument("--optc-eval", choices=["test", "none"], default="test")\n    ap.add_argument("--max-optc-test", type=int, default=0, help="chi de chay thu: gioi han so mau am OpTC TEST")\n    ap.add_argument("--no-amp", action="store_true")\n    ap.add_argument("--cpu", action="store_true")\n    args = ap.parse_args()\n    args.batch_size = args.batch_size or (32 if args.model == "bilstm" else 16)\n    args.eval_batch = args.eval_batch or (128 if args.model == "bilstm" else 32)\n    dev = torch.device("cpu" if args.cpu or not torch.cuda.is_available() else "cuda")\n    ft = Path(args.ft_root)\n    man = json.load(open(ft / "ft_manifest.json", encoding="utf-8"))\n    base = ft / "base_models" / args.model\n    cfg = json.load(open(base / "config.json", encoding="utf-8"))\n    thr = float(json.load(open(base / "threshold.json", encoding="utf-8"))["threshold"])\n    ck = torch.load(base / "best_model.pt", map_location="cpu", weights_only=False)\n    Path(args.out_dir).mkdir(parents=True, exist_ok=True)\n    print(f"[init] mo hinh {args.model}, thiet bi {dev}, nguong VAL OpTC goc {thr}, thu muc ra {args.out_dir}", flush=True)\n    optc_manifest = pd.read_csv(Path(args.optc_root) / "split/network_sample_manifest.csv")\n    rr = pd.read_csv(ft / "replay_R.csv")\n    rr["root"] = str(Path(args.optc_root))\n    t0 = time.time()\n    replay = T.materialize(rr)\n    print(f"[init] replay OpTC that: {len(replay)} mau (duong {sum(s[\'label\'] for s in replay)}) nap trong {time.time() - t0:.0f}s", flush=True)\n    ctx = {"device": dev, "cfg": cfg, "base_sd": ck["model_state_dict"], "thr": thr, "domains": man["domains"], "replay": replay, "optc_manifest": optc_manifest}\n    results = [run_fold(args, k, ctx) for k in [int(x) for x in args.folds.split(",")]]\n    json.dump({"model": args.model, "args": vars(args), "folds": [r["fold"] for r in results]}, open(Path(args.out_dir) / f"run_{args.model}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)\n    print("[xong]", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
print('Wrote:', sorted(p.name for p in WORK_CODE.iterdir()))
sys.path.insert(0, str(WORK_CODE))


## Chạy IL

In [ ]:
OUTPUT_DIR = Path(f'/kaggle/working/{RUN_NAME}')
cmd = (
    f"python {WORK_CODE/'s36_il_train.py'} --model {MODEL} "
    f"--ft-root '{FT_ROOT}' --optc-root '{DATASET_ROOT}' --code-dir '{WORK_CODE}' --out-dir '{OUTPUT_DIR}' "
    f"--folds {FOLDS} --seed {SEED} --optc-eval {OPTC_EVAL}"
)
print(cmd)
import subprocess
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f'exit code {ret}')


## Ma trận nhầm lẫn từng fold (ngưỡng chọn trên VAL)

In [ ]:
for p in sorted(OUTPUT_DIR.glob('result_*.json')):
    r = json.loads(p.read_text())
    print('=== fold', r['fold'], '| khớp khởi tạo (chênh logit):', r['equivalence_max_logit_diff'], '| tốt nhất:', r['best_stage'], 'epoch', r['best_epoch'])
    print('  ngưỡng (chọn trên VAL): logit', round(r['threshold_logit'], 3), '= prob', round(r['threshold_prob'], 4))
    for role in ('train', 'val', 'test'):
        x = r[f'atlasv2_{role}']
        print(f'  ATLASv2 {role:5s}:', x['n'], 'mẫu,', x['positives'], 'dương |', json.dumps(x['confusion']), '| PR-AUC', x.get('pr_auc'), 'ROC-AUC', x.get('roc_auc'))
    print('  TEST theo kịch bản:', json.dumps(r['atlasv2_test_by_scenario']))
    if 'optc_test' in r:
        o = r['optc_test']
        print('  OpTC TEST (ngưỡng VAL ATLASv2)   :', json.dumps(o['at_val_threshold']))
        print('  OpTC TEST (ngưỡng gốc OpTC)       :', json.dumps(o['at_optc_base_threshold']))


## Nén kết quả để tải về

In [ ]:
archive = shutil.make_archive(f'/kaggle/working/{RUN_NAME}', 'zip', OUTPUT_DIR)
print('Created:', archive)
